# SO-101 Phase 6A — MuJoCo Joint-Hold Experiments (Google Colab)

An offline SO-101 joint-position-hold benchmark using MuJoCo. This notebook fetches the selected GitHub branch, runs physical simulations, compares damping and time-integration settings, plots results, and optionally renders a 2×2 MP4 using the repository's existing offline visualization script.

No robot hardware is controlled. The baseline disables self-collision to isolate actuator dynamics; **this does not establish grasp performance or physical safety**. CPU runtime is sufficient for simulation. Movie rendering requires a working headless OpenGL backend; the saved Experiment 16 scene additionally requires private source data (kept outside GitHub).

**Comparison design:** all trials start from the same pose and change only one parameter from the baseline. The full set contains four runs: baseline, higher damping, smaller timestep, and implicitfast integrator. The baseline is a diagnostic reference, not a validated controller.


In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'mujoco>=3.3,<4', 'pandas', 'matplotlib', 'pillow',
                'scipy', 'pyarrow', 'opencv-python-headless'], check=True)
import mujoco
print('MuJoCo:', mujoco.__version__)


In [ ]:
from pathlib import Path
REPO=Path('/content/so101-imitation-learning')
REMOTE='https://github.com/neogim/so101-imitation-learning.git'
BRANCH='feature/so101-dynamic-arm-control'
if not REPO.exists():
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REMOTE,str(REPO)],check=True)
else:
    current=subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    assert current==BRANCH, f'Unexpected local branch: {current}'
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
print(subprocess.check_output(['git','-C',str(REPO),'log','-1','--oneline'],text=True))
assert (REPO/'data/urdf/so101/so101_new_calib.urdf').is_file()
assert (REPO/'data/urdf/so101/assets').is_dir()


## Controlled parameter comparison

The PD law in joint coordinates is

\[
\tau = K_p (q_{\mathrm{target}}-q) - K_d\dot{q}.
\]

\(K_p\) pulls the joint toward its target; \(K_d\) opposes joint velocity. Compare the results using the same 1-second experiment duration, fixed gravity, fixed actuator limits and disabled self-collision. The timestep comparison uses more integration steps to cover the same physical time.


In [ ]:
from datetime import datetime, timezone
import json, pandas as pd, numpy as np
STAMP=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR=Path('/content/phase6-results')/f'phase6a-compare-{STAMP}'
RUN_DIR.mkdir(parents=True,exist_ok=False)
SCRIPT=REPO/'scripts/so101_sim/phase6_dynamic_joint_hold.py'
TRIALS={
    'baseline_euler_2ms': {'kd':1.8,'dt':0.002,'integrator':'euler'},
    'damping_3p6_euler_2ms': {'kd':3.6,'dt':0.002,'integrator':'euler'},
    'euler_0p5ms': {'kd':1.8,'dt':0.0005,'integrator':'euler'},
    'implicitfast_2ms': {'kd':1.8,'dt':0.002,'integrator':'implicitfast'},
}
summaries={}
for label,params in TRIALS.items():
    out=RUN_DIR/label
    cmd=[sys.executable,'-u',str(SCRIPT),'--duration-s','1',
         '--self-collision','off','--kp','18','--kd',str(params['kd']),
         '--dt',str(params['dt']),'--integrator',params['integrator'],
         '--max-torque-nm','3','--out',str(out)]
    result=subprocess.run(cmd,cwd=REPO,text=True,capture_output=True)
    if result.returncode:
        print(result.stdout[-2000:],result.stderr[-3000:])
        raise RuntimeError(f'{label} failed (exit {result.returncode})')
    summaries[label]=json.loads((out/'summary.json').read_text())
    print(label, 'steps=',summaries[label]['physics_steps_completed'],
          'max joint error=',round(max(summaries[label]['max_abs_error_deg'].values()),2),'deg')
print('Results:',RUN_DIR)


In [ ]:
rows=[]
for label,s in summaries.items():
    r={'trial':label,'integrator':s['integrator'],'dt_ms':s['dt_s']*1000,
       'kd':s['kd_nms_per_rad'],'steps':s['physics_steps_completed'],
       'initial_contacts':s['initial_contacts_count']}
    for joint in ['shoulder_pan','shoulder_lift','elbow_flex','wrist_flex','wrist_roll','gripper']:
        r[joint+'_max_error_deg']=s['max_abs_error_deg'][joint]
    r['max_error_deg']=max(s['max_abs_error_deg'].values())
    r['max_speed_deg_s']=max(s['max_abs_joint_speed_deg_s'].values())
    rows.append(r)
comparison=pd.DataFrame(rows).set_index('trial')
display(comparison.round(2))
comparison.to_csv(RUN_DIR/'comparison.csv')


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(11,4))
for label in TRIALS:
    frame=pd.read_csv(RUN_DIR/label/'joint_trace.csv')
    ax.plot(frame['time_s'],frame['wrist_roll_error_deg'],label=label)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Wrist roll: target - actual (deg)')
ax.grid(True)
ax.legend()
plt.show()


## Optional: Save numerical results to Google Drive

The following cell mounts a personal Google Drive and copies numerical experiment outputs. It does not publish data to GitHub. Check access permissions and remove machine-specific absolute paths from metadata before sharing results elsewhere.


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DEST=Path('/content/drive/MyDrive/SO101_Phase6/hold_results')/RUN_DIR.name
DEST.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(RUN_DIR,DEST)
print('Saved:',DEST)


## Optional: Render the recorded motion as a 2×2 MP4

Uses `scripts/so101_sim/phase6_render_legacy_camera_trace.py` without changing the physics results. The left column shows the fixed reference pose; the right column replays the selected recorded actual joint positions. This is an offline visualization, **not** an additional physics simulation.

- **`bare`**: runs from public repository files only, but has the minimal unlit background.
- **`experiment16`**: uses the existing lighted, table-and-objects scene from Experiment 16. Its original episode dataset and camera/layout JSON files must be available privately in Google Drive. Place them under `MyDrive/SO101_Phase6/scene_inputs/` with the same relative paths used by the project: `data/so101/muhi-side-wrist-success-053-20261025-168x224/`, `outputs/real2sim/episode4-objects-20261108-004859.json`, `outputs/real2sim/episode4-cup-gui-20261108-131429.json`, `outputs/so101-eye-to-hand-id1-20261107/result.json`, and `assets/calibration/side_charuco_20261107_restricted_intrinsics_candidate.json`. The scene is only visual context and does not add contacts to the Phase6 simulation.

This section runs only when explicitly executed. For Experiment 16 mode, run the Google Drive mount cell above first.


In [ ]:
import os
from IPython.display import Video, display

RENDER_TRIAL='baseline_euler_2ms'  # also: damping_3p6_euler_2ms / euler_0p5ms / implicitfast_2ms
RENDER_SCENE='bare'  # change to 'experiment16' when private inputs are available
VIS=RUN_DIR/f'{RENDER_TRIAL}-four-view-{RENDER_SCENE}'
render_script=REPO/'scripts/so101_sim/phase6_render_legacy_camera_trace.py'
trace=RUN_DIR/RENDER_TRIAL/'joint_trace.csv'
cmd=[sys.executable,'-u',str(render_script),
     '--trace',str(trace),'--out',str(VIS),'--scene',RENDER_SCENE,
     '--stride','25','--fps','8','--video']
if RENDER_SCENE=='experiment16':
    private=Path('/content/drive/MyDrive/SO101_Phase6/scene_inputs')
    sources={
        '--dataset-root':'data/so101/muhi-side-wrist-success-053-20261025-168x224',
        '--object-layout-json':'outputs/real2sim/episode4-objects-20261108-004859.json',
        '--cup-pose-json':'outputs/real2sim/episode4-cup-gui-20261108-131429.json',
        '--extrinsics':'outputs/so101-eye-to-hand-id1-20261107/result.json',
        '--intrinsics':'assets/calibration/side_charuco_20261107_restricted_intrinsics_candidate.json',
    }
    missing=[str(private/relative) for relative in sources.values() if not (private/relative).exists()]
    if missing:
        raise FileNotFoundError('Experiment 16 inputs are missing from private Drive:\n'+'\n'.join(missing))
    for flag,relative in sources.items():
        cmd.extend([flag,str(private/relative)])
# EGL is suitable for GPU-enabled Linux runtimes. CPU-only Colab can use OSMesa
# when the system OpenGL software renderer is available.
os.environ['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
print('Renderer:',os.environ['MUJOCO_GL'])
subprocess.run(cmd,cwd=REPO,check=True,env=os.environ.copy())
mp4=VIS/'phase6_dls_sac_style_four_views.mp4'
if mp4.is_file():
    display(Video(str(mp4),embed=True))
else:
    print('MP4 not created; inspect the frames directory and ffmpeg availability:',VIS)


## Reproducibility and limitations

The model uses the repository URDF and simulated PD position actuators. Robot self-collision is disabled in the numerical comparisons. The comparison does not establish stable physical holding unless error and speed metrics support it. Results are not applicable to hardware without independent control-system validation.

The Experiment 16 visualization uses saved calibration-candidate data and a historical workcell; the scene is not synchronized with the object-free Phase 6A physical environment. Do not infer contact or grasp success from the MP4.
